In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

In [ ]:
df = pd.read_csv("corpus_df.csv", encoding="utf-8")  # replace with your actual file path

In [ ]:
custom_sw = german_sw + [  # historical forms
]

In [ ]:
docs = df["text"].astype(str).tolist()

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

hdbscan_model = HDBSCAN(
    min_cluster_size=7, ## experiment with this to increase/decrease number of topics
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=custom_sw,
    ngram_range=(1, 2))

umap_model = UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

topic_model = BERTopic(
    language="multilingual",
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    verbose=True
)


topics, probs = topic_model.fit_transform(docs)



In [ ]:
topic_info = topic_model.get_topic_info()
topic_model.get_topic_info()[["Topic", "Count", "Name"]]

In [ ]:
topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=False).head(10) ## check the largest topics for interpretability
#topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=True).head(10) ## check the smallest topics for potential noise

In [ ]:
fig = topic_model.visualize_topics()
fig.show()

In [ ]:
fig = topic_model.visualize_barchart()
fig.show()

fig = topic_model.visualize_heatmap()
fig.show()

fig = topic_model.visualize_hierarchy()
fig.show()

fig = topic_model.visualize_documents(docs)
fig.show()

In [ ]:
timestamps = df["date"]
timestamps = pd.to_datetime(df["date"])
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,
    nr_bins=20  # adjust depending on corpus size
)


In [ ]:
fig = topic_model.visualize_topics_over_time(topics_over_time)
fig.show()

In [ ]:
classes = df["author"]

In [ ]:
topics_per_class = topic_model.topics_per_class(
    docs,
    classes
)

In [ ]:
fig = topic_model.visualize_topics_per_class(topics_per_class)
fig.show()

In [ ]:
ToDO:

- check the largest topics for interpretability
- check the time evolution of topics
- have a way to give the top texts per topic for interpretation
